# Анализ `processors-am4.json`

Таблица: **имя**, **цена**, **ядер**, **потоков**, **частота** (ГГц), **value** = потоки × частота, **value/$** = value / цена (цена в валюте из JSON, обычно BYN).

Сортировка по убыванию **value/$**.

Зависимости: `pip install pandas` (в среде Jupyter).

Файл `processors-am4.json` ищется в текущей рабочей папке или в `onliner.by-crawler/` (удобно, если kernel стартовал из корня репозитория).

In [10]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CANDIDATES = [
    ROOT / "processors-am4.json"
]
DATA = next((p for p in CANDIDATES if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Не найден processors-am4.json. Положите его рядом с ноутбуком или запустите краулер (run-search.bat)."
    )

with open(DATA, encoding="utf-8") as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
needed = ["name", "price", "coreCount", "threadCount", "maxFrequencyGHz"]
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise ValueError(f"В JSON нет колонок: {missing_cols}")

out = df[needed].copy()
# Если threadCount не заполнен, используем coreCount.
out["threadCount"] = out["threadCount"].fillna(out["coreCount"])
out.columns = ["имя", "цена", "ядер", "потоков", "частота"]
out["value=потоков*частота"] = out["потоков"] * out["частота"]

price_num = pd.to_numeric(
    out["цена"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)
out["value/руб"] = out["value=потоков*частота"].div(price_num).where(price_num > 100)

out_sorted = out.sort_values("value/руб", ascending=False, na_position="last")
out_sorted.head(12)

,имя,цена,ядер,потоков,частота,value=потоков*частота,value/руб
12,AMD Ryzen 3 3100,104.42,4,8.0,3.9,31.2,0.298793
32,AMD Ryzen 5 4500,245.13,6,12.0,4.1,49.2,0.200710
35,AMD Ryzen 5 5500,255.26,6,12.0,4.2,50.4,0.197446
29,AMD Ryzen 5 3600,262.05,6,12.0,4.2,50.4,0.192330
61,AMD Ryzen 7 5700,476.36,8,16.0,4.6,73.6,0.154505
66,AMD Ryzen 7 5700X,496.10,8,16.0,4.6,73.6,0.148357
59,AMD Ryzen 7 3700X,496.51,8,16.0,4.4,70.4,0.141790
24,AMD Ryzen 5 3400G,237.32,4,8.0,4.2,33.6,0.141581
41,AMD Ryzen 5 5600,374.66,6,12.0,4.4,52.8,0.140928
53,AMD Ryzen 5 Pro 3600,364.47,6,12.0,4.2,50.4,0.138283


In [11]:
# Строки без потоков или частоты (value не посчитать)
mask = out["потоков"].isna() | out["частота"].isna()
if mask.any():
    display(out.loc[mask])
else:
    print("Все строки имеют потоки и частоту для расчёта value.")

Все строки имеют потоки и частоту для расчёта value.
